# Notebook 8: Model Evaluation & Interpretability
In this notebook, we will:
1. Compare the evaluation metrics of all trained models (Naive Bayes, Logistic Regression, Linear SVM, Random Forest, XGBoost).
2. Inspect the **feature importances / coefficients** of our Logistic Regression model to see which words are most associated with fraudulent job postings (Interpretability).

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

# Let's construct a comparative dataframe of all our models' results on the test set:
results = {
    "Model": ["Dummy Classifier", "Naive Bayes", "Logistic Regression", "Linear SVM", "Random Forest", "XGBoost"],
    "Accuracy": [0.9516, 0.9684, 0.9586, 0.9832, 0.9653, 0.9818],
    "Precision": [0.0000, 0.8488, 0.5433, 0.8192, 0.6061, 0.8462],
    "Recall": [0.0000, 0.4220, 0.9075, 0.8382, 0.8092, 0.7630],
    "F1-Score": [0.0000, 0.5637, 0.6797, 0.8286, 0.6931, 0.8024],
    "ROC-AUC": [0.5000, 0.9382, 0.9892, 0.9865, 0.9843, 0.9780]
}

df_results = pd.DataFrame(results)
print("=== Comparative Model Experiments ===")
print(df_results.to_string(index=False))

=== Comparative Model Experiments ===
              Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
   Dummy Classifier    0.9516     0.0000  0.0000    0.0000   0.5000
        Naive Bayes    0.9684     0.8488  0.4220    0.5637   0.9382
Logistic Regression    0.9586     0.5433  0.9075    0.6797   0.9892
         Linear SVM    0.9832     0.8192  0.8382    0.8286   0.9865
      Random Forest    0.9653     0.6061  0.8092    0.6931   0.9843
            XGBoost    0.9818     0.8462  0.7630    0.8024   0.9780


## 1. Model Interpretability: Top Words associated with Fraud
We will load our saved Logistic Regression model and TF-IDF Vectorizer to extract the model's coefficients. Coefficients represent the weight/importance given to each word by the model. A highly positive coefficient means the presence of the word strongly points to **Fraudulent** (1), while a highly negative coefficient points to **Legitimate** (0).

In [2]:
model_path = os.path.join("..", "models", "logistic_regression.pkl")
vectorizer_path = os.path.join("..", "models", "tfidf_vectorizer.pkl")

lr = joblib.load(model_path)
vectorizer = joblib.load(vectorizer_path)

# Extract features/words
words = list(vectorizer.get_feature_names_out())

# Add name of numerical features which we tacked onto the end of TF-IDF matrix
feature_names = words + ['telecommuting', 'has_company_logo', 'has_questions']

# Get model coefficients
coefs = lr.coef_[0]

# Create dataframe of word weights
weights_df = pd.DataFrame({
    'Feature': feature_names,
    'Weight': coefs
})

# Top 20 Fraud Indicators
print("=== Top 20 Words Indicating FRAUD ===")
print(weights_df.sort_values(by='Weight', ascending=False).head(20).to_string(index=False))

print("\n=== Top 20 Words Indicating LEGITIMATE Jobs ===")
print(weights_df.sort_values(by='Weight', ascending=True).head(20).to_string(index=False))

=== Top 20 Words Indicating FRAUD ===
                 Feature   Weight
             high school 3.589926
                aptitude 3.578646
                    link 3.490629
                    earn 3.286524
              data entry 3.022605
                   money 2.980063
                   phone 2.872814
              encouraged 2.867967
                 oil gas 2.840044
              leveraging 2.777091
                 signing 2.725389
            receptionist 2.606057
    compensation package 2.546604
                     gas 2.537702
                      cf 2.526910
            facilitating 2.499308
administrative assistant 2.483293
                      ae 2.465905
                  school 2.448308
                offshore 2.445217

=== Top 20 Words Indicating LEGITIMATE Jobs ===
         Feature    Weight
         english -2.630727
         growing -2.343793
             fun -2.237009
     recruitment -2.211434
has_company_logo -2.188463
           based -2.099158
          